# ESdE Adultos 2023: Test Exploratory analysis

~~This notebook is for exploratory work with the extracted INE microdata. Reusable extraction, loading, and codebook functions stay in `ine_health_data/`; calculations and visual inspection belong here.~~

Run the notebook from the repository root. 

The raw ESdE file and `references/metadata/esde_adulto_2023.json` must already exist. Otherwise run `ine_health_data.pipeline.start_sequence()`

In [1]:
import pandas as pd
from ine_health_data.pipeline import add_value_labels, load_variables

## General population analysis
(population represented)

Stablishing the population that the dataset represents before realizing any in depth analysis.

Main target variables should be age, sex, and location.

In [2]:
analysis_variable = ["EDADa","SEXOa","CCAA"] 
raw_data = load_variables(variables=analysis_variable)

### Data integrity

The variable `EDADa` is the only numeric variable loaded, which includes special `999` code for "Not answered" value.
For `SEXOa`,`CCAA`, There are no possible "Not answered" special values.

In [3]:
rows_count          = len(raw_data)
nonfull_rows_count  = raw_data.isna().any(axis=1).sum()
age_not_answered_c  = raw_data["EDADa"].eq(999).sum()

pd.Series({
    "Total rows": rows_count,
    "Rows any with empty": nonfull_rows_count,
    "Not answered age rows": age_not_answered_c
}).to_frame(name="n")

,n
Total rows,21032
Rows any with empty,0
Not answered age rows,0


The dataset for `SEXOa`, `CCAA` and `EDADa` does **not** contain any empty or "Not answered" value.

This implication is carried into following python cells, avoiding the need for unnecessary filtering.

### Age and Sex analysis

#### General Age

In [4]:
non_ccaa_df = raw_data[["EDADa","SEXOa"]]

age_summary = (
    non_ccaa_df.loc[non_ccaa_df["EDADa"].ne(999), "EDADa"]
    .describe(percentiles=[0.25,0.5,0.75])
    .to_frame().T
    .rename(index={"EDADa": "General age data"})
    .round(2)
)
display(age_summary)


,count,mean,std,min,25%,50%,75%,max
General age data,21032.0,54.49,19.14,15.0,40.0,55.0,69.0,103.0


#### Age by Sex

In [5]:
labeled_national_df = add_value_labels(non_ccaa_df)

age_by_sex_freq = labeled_national_df["SEXOa_label"].value_counts(normalize=True).round(4)
age_by_sex_desc = (
    labeled_national_df.groupby("SEXOa_label")["EDADa"]
    .describe(percentiles=[0.25,0.5,0.75])
    .round(2)
)
age_by_sex_summary = (
    pd.concat([age_by_sex_freq, age_by_sex_desc], axis=1)
    .rename(columns={"SEXOa_label": "sex"}).rename_axis("age by sex")
)
display(age_by_sex_summary)

,proportion,count,mean,std,min,25%,50%,75%,max
age by sex,,,,,,,,,
Mujer,0.5397,11352.0,55.92,19.49,15.0,41.0,56.0,71.0,103.0
Hombre,0.4603,9680.0,52.8,18.59,15.0,39.0,53.0,67.0,98.0


#### Age and Sex per Location

In [6]:
df_labeled = add_value_labels(raw_data)

statistics = ["n", "mean", "median"]
groups = ["Total", "Hombre", "Mujer"]

total_by_ccaa = (
    df_labeled.groupby(["CCAA_label"])["EDADa"]
    .describe()
    .round(2)
    .drop(columns=["std","min","max","25%","75%"])
    .rename(columns={"50%":"median","count":"n"})
)
total_by_ccaa.columns = pd.MultiIndex.from_product(
    [total_by_ccaa.columns, ["Total"]],
    names=["Estadística","Sexo",],
)

age_by_sex_ccaa = (
    df_labeled.groupby(["CCAA_label","SEXOa_label"])["EDADa"]
    .describe() 
    .drop(columns=["std","min","max","25%","75%"])
    .unstack("SEXOa_label")
    .round(2)
    .rename(columns={"50%":"median","count":"n"})
)

summary = (
    pd.concat([total_by_ccaa, age_by_sex_ccaa], axis=1)
    .sort_values(("n","Total"), ascending=False)
    .rename_axis("Age and Sex per CCAA")
) 
summary = summary.reindex(
    columns=pd.MultiIndex.from_product(
        [statistics, groups],
        names=["Statistic", "Sex"],
    )
)
display(summary)


Statistic                         n                   mean                \
Sex                           Total  Hombre   Mujer  Total Hombre  Mujer   
Age and Sex per CCAA                                                       
Andalucía                    2674.0  1239.0  1435.0  52.67  50.51  54.53   
Comunitat Valenciana         2059.0   928.0  1131.0  54.55  52.66  56.11   
Madrid, Comunidad de         1876.0   846.0  1030.0  53.82  51.51  55.73   
Cataluña                     1802.0   821.0   981.0  54.32  53.16   55.3   
País Vasco                   1364.0   619.0   745.0   55.8  53.52  57.69   
Castilla y León              1344.0   693.0   651.0  57.05  56.32  57.83   
Extremadura                  1099.0   541.0   558.0  55.65  53.19  58.03   
Murcia, Región de            1080.0   486.0   594.0  52.34   49.3  54.82   
Aragón                       1044.0   464.0   580.0  55.73  54.42  56.78   
Castilla - La Mancha         1030.0   500.0   530.0  55.31  54.05  56.49   
Galicia                       904.0   399.0   505.0  56.48   55.0  57.65   
Canarias                      874.0   407.0   467.0   52.3  51.51  52.99   
Navarra, Comunidad Foral de   854.0   396.0   458.0  55.36  54.35  56.23   
Asturias, Principado de       705.0   316.0   389.0  58.16  56.65   59.4   
Balears, Illes                680.0   283.0   397.0  52.59  49.94  54.48   
Cantabria                     635.0   291.0   344.0  55.45  53.87  56.79   
Rioja, La                     579.0   257.0   322.0  54.22  54.24   54.2   
Melilla                       226.0   100.0   126.0  48.17  48.12  48.21   
Ceuta                         203.0    94.0   109.0  49.22  46.72  51.37   

Statistic                   median               
Sex                          Total Hombre Mujer  
Age and Sex per CCAA                             
Andalucía                     53.0   49.0  55.0  
Comunitat Valenciana          55.0   53.0  56.0  
Madrid, Comunidad de          53.0   51.0  55.0  
Cataluña                      55.0   54.0  56.0  
País Vasco                    56.0   54.0  58.0  
Castilla y León               58.0   59.0  58.0  
Extremadura                   56.0   54.0  59.0  
Murcia, Región de             53.0   48.5  57.0  
Aragón                        56.0   54.0  58.0  
Castilla - La Mancha          55.0   54.0  57.0  
Galicia                       57.0   56.0  58.0  
Canarias                      53.0   53.0  53.0  
Navarra, Comunidad Foral de   56.0   56.0  56.0  
Asturias, Principado de       59.0   58.0  60.0  
Balears, Illes                52.0   49.0  54.0  
Cantabria                     56.0   55.0  59.0  
Rioja, La                     56.0   57.0  55.0  
Melilla                       46.5   46.0  47.0  
Ceuta                         48.0   46.5  54.0